# 🚗 Car Component Detection — Training on Google Colab (T4 GPU)

Notebook ini digunakan untuk melatih model **Multi-Scale PA-SCAN (Custom CNN from Scratch)** menggunakan GPU T4 gratis di Google Colab.

### ⚠️ Langkah Awal:
Pastikan Runtime sudah diatur ke GPU:
**Runtime** -> **Change runtime type** -> Pilih **T4 GPU** -> **Save**.

In [15]:
# 1. Periksa Akses GPU Colab
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
else:
    print("⚠️ GPU BELUM AKTIF! Ubah runtime ke GPU di menu Runtime -> Change runtime type.")

PyTorch: 2.14.1+cu130 | CUDA: True
GPU Aktif: Tesla T4 (15.6 GB VRAM)


## 2. Ekstrak File .ZIP Otomatis

Cell di bawah ini akan otomatis mendeteksi file zip mana pun yang ada di Colab (`car-detection-clean.zip` atau `car-detection.zip`) lalu mengekstraknya menggunakan Python bawaan.

In [8]:
import zipfile
import os

# Cari file zip yang diupload
zip_names = ["car-detection-clean.zip", "car-detection.zip"]
found_zip = None
for z in zip_names:
    if os.path.exists(z):
        found_zip = z
        break

if found_zip:
    print(f"Sedang mengekstrak {found_zip}...")
    with zipfile.ZipFile(found_zip, 'r') as zip_ref:
        zip_ref.extractall("car-detection")
    print("✅ Ekstraksi selesai! Folder 'car-detection' siap digunakan.")
else:
    print("⚠️ File zip belum ditemukan! Pastikan Anda sudah mengupload 'car-detection-clean.zip' ke tab Files di panel kiri.")

Sedang mengekstrak car-detection-clean.zip...
✅ Ekstraksi selesai! Folder 'car-detection' siap digunakan.


## 3. Masuk ke Folder Project & Install Dependensi Ringan

In [7]:
ls -l

total 71258
-rw------- 1 root root 72967983 Nov 14 15:27 car-detection-clean.zip


In [11]:
%cd car-detection
!pip install -q scikit-learn pillow albumentations
print("✅ Dependensi siap!")

/content/drive/MyDrive/car-detection/car-detection
✅ Dependensi siap!


## 4. Jalankan Training (GPU Accelerated)

Script training sudah dirancang otomatis menggunakan CUDA dan FP16 AMP. Di GPU T4 Colab, 40 epoch selesai dalam **~20 detik**!

In [12]:
!python model/train.py --epochs 50 --batch-size 16 --lr 0.0025

OS: Linux (6.6.159+) | Device: cuda [Tesla T4, 14.6 GB VRAM] | CUDA: 13.0 | AMP FP16: True
Model parameters: 7,498,591 | Training on: /content/drive/MyDrive/car-detection/car-detection/dataset/raw
Dataset → train: 192 | val: 64 | workers: 1 (Linux) (val is angle-stratified, clean ROI, 50/50 per-label)
pos_weight per label:
  front_left_door: 0.959
  front_right_door: 1.000
  rear_left_door: 1.000
  rear_right_door: 1.000
  hood: 1.000
Epoch 01/50 ( 8.3s) | BCE: 0.7138 | Val: 0.7040 | Mean: 52.5% | Exact: 4.7% | FL:62.5% | FR:50.0% | RL:50.0% | RR:50.0% | Hood:50.0%
  >> Saved best model (mean_acc=52.50%, val_loss=0.7040)
Epoch 02/50 ( 1.2s) | BCE: 0.6637 | Val: 0.7306 | Mean: 53.8% | Exact: 6.2% | FL:59.4% | FR:57.8% | RL:50.0% | RR:51.6% | Hood:50.0%
  >> Saved best model (mean_acc=53.75%, val_loss=0.7306)
Epoch 03/50 ( 1.3s) | BCE: 0.6276 | Val: 0.7009 | Mean: 51.9% | Exact: 3.1% | FL:45.3% | FR:53.1% | RL:56.2% | RR:50.0% | Hood:54.7%
Epoch 04/50 ( 1.2s) | BCE: 0.5271 | Val: 0.6507 

## 5. Evaluasi Model (Menghitung Precision, Recall, F1, & ROC-AUC)

In [13]:
!python model/evaluate.py

Evaluating on OS: Linux | Device: cuda (Tesla T4)
Dataset → train: 192 | val: 64 | workers: 1 (Linux) (val is angle-stratified, clean ROI, 50/50 per-label)
PER-LABEL REPORT  (threshold = 0.50)

-- FRONT_LEFT_DOOR --
              precision    recall  f1-score   support

      Closed       1.00      1.00      1.00        32
        Open       1.00      1.00      1.00        32

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64

  ROC-AUC : 1.0000

-- FRONT_RIGHT_DOOR --
              precision    recall  f1-score   support

      Closed       1.00      1.00      1.00        32
        Open       1.00      1.00      1.00        32

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64

  ROC-AUC : 1.0000

-- REAR_LEFT_DOOR --
              precision    recall  f1-score   

## 6. Download Model Hasil Training ke PC Lokal

Jalankan cell ini untuk mengunduh bobot `best_model.pt` dan `thresholds.json`. Setelah terdownload, masukkan kedua file ini ke folder `checkpoints/` di PC lokal Anda.

In [14]:
from google.colab import files
import os

if os.path.exists("checkpoints/best_model.pt"):
    print("Mengunduh best_model.pt...")
    files.download("checkpoints/best_model.pt")

if os.path.exists("checkpoints/thresholds.json"):
    print("Mengunduh thresholds.json...")
    files.download("checkpoints/thresholds.json")

Mengunduh best_model.pt...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mengunduh thresholds.json...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>